In [0]:
df_date = spark.read.table("ecommerce.bronze.brz_date")
df_date.printSchema()

In [0]:
df_date.show(5)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
df_date = df_date.withColumn("date", to_date(col("date"), 'dd-MM-yyyy')) \
        .withColumn("day_name", initcap(col("day_name"))) \
        .withColumn("week_of_year", abs(col("week_of_year")).cast(IntegerType()))


In [0]:
df_date.show(4)

In [0]:
col_mapping = {
    "date" : "Date",
    "year" : "Year",
    "day_name" : "Day",
    "quarter" : "Quarter",
    "week_of_year" : "Week_Of_Year"
}

for old_name, new_name in col_mapping.items():
    df_date = df_date.withColumnRenamed(old_name, new_name)

df_date.show(5)

In [0]:
# make 3 -> Q3 - YEAR , WOY WEEK<> - Year 
from pyspark.sql.functions import *
df_date = df_date.withColumn("Quarter", concat(lit("Q"), col("Quarter"), lit("-"), col("Year"))) \
    .withColumn("Week_Of_Year", concat(lit("WEEK"), col("Week_Of_Year"), lit("-"), col("Year")))

df_date.show(10)

### Write to S3 and Silver 

In [0]:
df_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","True") \
    .save("s3://sj-dbr-demo-proj/silver_data/slv_date")

spark.sql("""
          create table if not exists ecommerce.silver.slv_date
          using delta location 's3://sj-dbr-demo-proj/silver_data/slv_date'
          """)

In [0]:
for each in dbutils.fs.ls("/Volumes/ecommerce/raw/raw_aws_data"):
    print(each.name)

In [0]:
%skip
#Suppose you want to write simple pythin that returns custom object type as return type 
import os
from types import SimpleNamespace
from datetime import datetime

def get_file_info(filepath):
    # Check if file exists
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"{filepath} does not exist")
    
    # Get file stats
    stats = os.stat(filepath)
    
    # Create object with attributes
    file_info = SimpleNamespace(
        name=os.path.basename(filepath),
        path=os.path.abspath(filepath),
        size=stats.st_size,
        modification_time=datetime.fromtimestamp(stats.st_mtime),
        # Note: owner info is platform-dependent
        owner=getattr(os, "getlogin", lambda: "unknown")()
    )
    
    return file_info

# Example usage
f = get_file_info("example.txt")
print(f.name)
print(f.path)
print(f.size)
print(f.modification_time)
print(f.owner)

In [0]:
dbutils.notebook.exit("SUCCESS")